# 06 — Confluency evidence v2: fine-tuning dev runs + the sealed dense test (mCellSeg)

**What this runs (GPU only; scoring happens later on the Mac, once):**

- **Part A — development (MSC, already-scored images).** Fine-tune Cellpose-SAM on 20 images from two donor
  populations and read the third, for each of the 3 populations. Also zero-shot and zero-shot rescaled maps.
  Checks the fine-tuning recipe; not a held-out result.
- **Part B — the sealed test (mCellSeg, never read by any model).** Pre-registered in
  `results/confluency_mcellseg.md`, committed before this notebook was written. Zero-shot and rescaled maps,
  three fine-tuning runs (two cross-fit folds + final), fine-tuned maps.
- **Part C — SAMCell on the MSC images** (development comparison; last, because it installs extra packages).

Each part ends by reducing its maps to small reading files and copying **one results zip** to Drive.

**I/O rule.** Nothing is read through the Drive mount: the bundle zip is copied to local disk and unzipped;
mCellSeg is downloaded from Zenodo straight to local disk (MD5-checked).

**Before running:** upload `confluency_v2_bundle.zip` to `MyDrive/cultureqc/staged/`.

**Runtime:** A100 about 1.5–2.5 h in total; L4 roughly twice that. If the session drops, re-run Setup and
then only the part that did not finish (earlier parts' results are already on Drive).

**Bring back:** `MyDrive/cultureqc/staged/confluency_v2_results.zip` (a few MB).

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, time, subprocess, json, hashlib, zipfile
DRIVE_ROOT = '/content/drive/MyDrive/cultureqc'
STAGE_DIR = f'{DRIVE_ROOT}/staged'
BUNDLE = f'{STAGE_DIR}/confluency_v2_bundle.zip'
RESULTS_ZIP = f'{STAGE_DIR}/confluency_v2_results.zip'
EXPECTED_COMMIT = '001a256'        # the pre-registration commit the bundle was made from
REPO = '/content/cultureqc'
assert os.path.exists(BUNDLE), f'upload the bundle first: {BUNDLE}'
print(f'{BUNDLE}: {os.path.getsize(BUNDLE)/1e6:.0f} MB')

In [ ]:
t0 = time.time()
shutil.rmtree(REPO, ignore_errors=True)
shutil.copy(BUNDLE, '/content/bundle.zip')                 # one sequential read from Drive
!unzip -q -o /content/bundle.zip -d $REPO
os.remove('/content/bundle.zip')
%cd $REPO
commit = open('COMMIT').read().strip()
print('bundle commit', commit, f'— unpacked in {time.time()-t0:.0f} s')
assert commit.startswith(EXPECTED_COMMIT), (commit, EXPECTED_COMMIT)
!ls data/sources/msc/images | wc -l

In [ ]:
!pip install -q cellpose==4.2.1.1 tifffile scikit-image opencv-python-headless pyyaml
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU (A100 or L4)'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# mCellSeg (Alam et al. 2026, CC BY 4.0) straight from Zenodo to local disk, MD5 from the record
t0 = time.time()
os.makedirs('/content/dl', exist_ok=True)
!curl -sL -o /content/dl/mCellSeg.zip "https://zenodo.org/api/records/20174259/files/mCellSeg.zip/content"
md5 = hashlib.md5(open('/content/dl/mCellSeg.zip', 'rb').read()).hexdigest()
print('md5', md5)
assert md5 == 'ed06d6e4c10e93b81703984cef852246'
os.makedirs('data/sources/mcellseg', exist_ok=True)
!unzip -q -o /content/dl/mCellSeg.zip -d data/sources/mcellseg
os.remove('/content/dl/mCellSeg.zip')
!ls data/sources/mcellseg/mCellSeg/labeled/images | wc -l
print(f'{time.time()-t0:.0f} s')

In [ ]:
# Masks only: the split must match the one committed with the pre-registration
!python scripts/confluency_mcellseg.py count

### ⏸ Checkpoint
The last line above must say **`split matches the committed results/confluency_mcellseg_split.json`**
(and the table: 90 test images, 22 at 60–90%, 4 at ≥ 80%). If it says *SPLIT DIFFERS*, stop and send me the
output — don't run Part B.

In [ ]:
def save_results(label):
    # one small zip to Drive: reading files + run manifests + logs (no maps, no weights)
    keep = ['results/confluency_finetune_dev_curves.json', 'results/confluency_mcellseg_curves.json']
    keep += [os.path.join(r, f) for r, _, fs in os.walk('cache/finetune') for f in fs if f == 'manifest.json']
    keep += [os.path.join('logs', f) for f in sorted(os.listdir('logs'))] if os.path.isdir('logs') else []
    keep += ['cache/probmaps_alt/samcell/manifest.json']
    local = '/content/confluency_v2_results.zip'
    with zipfile.ZipFile(local, 'w', zipfile.ZIP_DEFLATED) as z:
        z.writestr('COMMIT', commit)
        for p in keep:
            if os.path.exists(p):
                z.write(p)
    shutil.copy(local, RESULTS_ZIP)
    print(label, '->', RESULTS_ZIP, f'{os.path.getsize(local)/1e6:.1f} MB')
    with zipfile.ZipFile(local) as z:
        for n in z.namelist(): print('  ', n)

os.makedirs('logs', exist_ok=True)
def run(cmd, log):
    # streams the output (minus progress bars) and keeps a full log; stops the cell if the step fails
    t0 = time.time()
    r = subprocess.run(['bash', '-c', f'set -o pipefail; {cmd} 2>&1 | tee -a logs/{log}.log | (grep -v -E "it/s]|s/it]" || true)'])
    print(f'[{log}] {(time.time()-t0)/60:.1f} min, exit {r.returncode}')
    if r.returncode != 0:
        save_results(f'{log} FAILED (partial results)')
        raise RuntimeError(f'{log} failed — send me logs/{log}.log (it is in the results zip)')

## Part A — development runs on the MSC images (~30–45 min on A100)

In [ ]:
for h in ['218-4', '218-5', '218-6']:
    run(f'python scripts/confluency_finetune.py train --holdout {h} --n 20', f'dev_train_{h}')
    run(f'python scripts/confluency_finetune.py maps --run msc_{h}_n20_s0', f'dev_maps_{h}')
run('python scripts/confluency_finetune.py maps --run base', 'dev_maps_base')
run('python scripts/confluency_finetune.py maps --run base_d89', 'dev_maps_base_d89')
run('python scripts/confluency_finetune.py curves', 'dev_curves')
save_results('Part A')

## Part B — the sealed test (~1–1.5 h on A100)

Order matters only for disk: maps first, then the three fine-tuning runs, then the fine-tuned maps.
Nothing here prints a score; scoring happens once, later, from the reading file.

In [ ]:
run('python scripts/confluency_mcellseg.py maps', 'sealed_maps')
run('python scripts/confluency_mcellseg.py train', 'sealed_train')
run('python scripts/confluency_mcellseg.py ftmaps', 'sealed_ftmaps')
run('python scripts/confluency_mcellseg.py curves', 'sealed_curves')
save_results('Part B')

## Part C — SAMCell on the MSC images (development comparison, ~15 min)

Installs SAMCell (MIT) and its weights from the authors' GitHub release; runs last because it adds packages.

In [ ]:
!pip install -q samcell==1.2.0 matplotlib
os.makedirs('cache/alt_models', exist_ok=True)
!curl -sL -o cache/alt_models/samcell-generalist.pt https://github.com/saahilsanganeriya/SAMCell/releases/download/v1/samcell-generalist.pt
run('python scripts/alt_model_samcell.py', 'samcell_maps')
run('python scripts/confluency_finetune.py curves', 'dev_curves_with_samcell')
save_results('Part C')

## Done
Download `MyDrive/cultureqc/staged/confluency_v2_results.zip` and put it in the repo folder (or tell me
where it is). Optional, for the record: the fine-tuned weights are in `cache/finetune/*/models/` on this
runtime; their SHA-256 values are in the manifests inside the zip, so they don't need to come back.